In [ ]:
import * as tslab from "tslab";
import { readFileSync } from "fs";

const css = readFileSync("../style.css", "utf-8");
tslab.display.html(`<style>${css}</style>`);

# Merge Sort

In order to sort an array $L$ using <em style="color:blue;">merge sort</em> we proceed as follows:

We need to specify how two sorted arrays $L_1$ and $L_2$ are merged in a way that the resulting
array is also sorted.

 - If the array $L_1$ is empty, the result is $L_2$: 
   $$ \mathtt{merge}([], L_2) = L_2 $$
 - If the array $L_2$  is empty, the result is $L_1$: 
   $$ \mathtt{merge}(L_1, []) = L_1 $$
 - Otherwise, $L_1$ must have the form $[x_1] + R_1$ and $L_2$ has the form $[x_2] + R_2$.
   Then there is a case distinction with respect to the comparison of $x_1$ and $x_2$:
   - $x_1 \leq x_2$.
     In this case, we merge $R_1$ and $L_2$ and put $x_1$ at the beginning of this array:
     $$x_1 \leq x_2 \rightarrow \mathtt{merge}\bigl([x_1] + R_1, [x_2] + R_2\bigr) = 
     \bigl[x_1\bigr] + \mathtt{merge}\bigl(R_1,[x_2] + R_2\bigr)
     $$
   - $\neg (x_1 \leq x_2)$.
     In this case, we merge $L_1$ and $R_2$ and put $x_2$ at the beginning of this array:
     $$\neg (x_1 \preceq x_2) \rightarrow \mathtt{merge}\bigl([x_1] + R_1, [x_2] + R_2\bigr) 
       = \bigl[x_2 \bigr] + \mathtt{merge}\bigl([x_1] + R_1, R_2\bigr)
     $$

In [ ]:
function merge(L1: number[], L2: number[]): number[] {
  if (L1.length === 0) {
    return L2;
  }
  if (L2.length === 0) {
    return L1;
  }

  const [x1, ...R1] = L1;
  const [x2, ...R2] = L2;

  if (x1 <= x2) {
    return [x1, ...merge(R1, L2)];
  } else {
    return [x2, ...merge(L1, R2)];
  }
}

 - If $L$ has less than two elements, then $L$ is already sorted.  Therefore we have: 
   $$ \#L < 2 \rightarrow \mathtt{sort}(L) = L $$
 - Otherwise, the array $L$ is split into two arrays that have approximately the same size.
   These arrays are sorted recursively.  Then, the sorted arrays are merged in a way that the
   resulting array is sorted: 
   $$ \#L \geq 2 \wedge \mathtt{n} := \#L \rightarrow \mathtt{sort}(L) =
         \mathtt{merge}\bigl(\mathtt{sort}\bigl(\texttt{L[:n//2]}\bigr),
         \mathtt{sort}\bigl(\texttt{L[n//2:]}\bigr)\bigr)
   $$
   Here, $\texttt{L[:n//2]}$ is the first part of the array, while
   $\texttt{L[n//2:]}$ is the second part.  If the length of $L$ is even, both part have the same    number of elements, otherwise the second part has one element more than the first part. 

In [ ]:
function sort(L: number[]): number[] {
  const n = L.length;
  if (n < 2) {
    return L;
  }
  const middle = Math.floor(n / 2);
  return merge(sort(L.slice(0, middle)), sort(L.slice(middle)));
}

In [ ]:
sort([7, 8, 11, 12, 2, 5, 3, 7, 9, 3, 2])

## Testing

The function `counter` takes an array as input and returns a Map that keeps count of how many times each item occurs in the array.

In [ ]:
function counter<T>(arr: T[]): Map<T, number> {
  const counts = new Map<T, number>();
  for (const item of arr) {
    counts.set(item, (counts.get(item) ?? 0) + 1);
  }
  return counts;
}

In [ ]:
console.log(counter(['a', 'b', 'a', 'b', 'c', 'a']));

We also define the helper function `compareCounter` to be able to compare the contents of two counters.

In [ ]:
function compareCounters(a: Map<number, number>, b: Map<number, number>): boolean {
  if (a.size !== b.size) return false;
  for (const [key, value] of a) {
    if (b.get(key) !== value) return false;
  }
  return true;
}

In [ ]:
function demo() {
  const L: number[] = Array.from({ length: 19 }, () => Math.floor(Math.random() * 99) + 1);
  console.log("L =", L);

  let S = [...L];
  S = sort(S);
  console.log("S =", S);

  const counterL = counter(L);
  const counterS = counter(S);

  console.log(counterL);
  console.log(counterS);
  const equal = compareCounters(counterL, counterS);
  console.log(equal);
}

In [ ]:
demo();

The function `isOrdered(L)` checks that the array `L` is sorted ascendingly.

In [ ]:
function isOrdered(L: number[]): void {
  for (let i = 0; i < L.length - 1; i++) {
    if (L[i] >= L[i + 1]) {
      throw new Error(`${L} not ordered at ${i}`);
    }
  }
}

The function `sameElements(L, S)` returns `true` if the array `L` and `S` contain the same elements and, furthermore, each 
element $x$ occurring in `L` occurs in `S` the same number of times it occurs in `L`.

In [ ]:
import assert from 'assert';

function sameElements(L: number[], S: number[]): void {
  assert(compareCounters(counter(L), counter(S)), "L and S do not have the same elements");
}

The function $\texttt{testSort}(n, k)$ generates $n$ random arrays of length $k$, sorts them, and checks whether the output is sorted and contains the same elements as the input.

In [ ]:
function testSort(n: number, k: number): void {
  for (let i = 0; i < n; i++) {
    const L: number[] = Array.from({ length: k }, () => Math.floor(Math.random() * (2 * k)));
    const S = sort(L);
    isOrdered(S);
    sameElements(L, S);
    process.stdout.write('.');
  }
  console.log();
  console.log("All tests successful!");
}

In [ ]:
console.time();
testSort(100, 2000);
console.timeEnd();